# One Page, One Finding — Titanic
## Could Jack have been saved?

**Group 1:** Kleant Laboti · *[Group member 2]* · *[Group member 3]*
**Dataset:** Kaggle Titanic competition — `train.csv` (891 passengers)
**Tools:** Python, pandas, NumPy, Matplotlib. No machine-learning models are used; this is a descriptive investigation.

---

### How to read this notebook
This is a research report, not a tidy final answer. It keeps the hypotheses we wrote *before*
looking at outcomes, the charts we discarded, the data problems we found, and the uncertainty we could
not remove. It runs top to bottom from a fresh kernel.

**Structure (the seven required sections):**
1. Load & profile the dataset
2. Three hypotheses, written before exploration
3. Cleaning, with a reason for every decision
4. Exploration — distributions, then relationships
5. Testing the three hypotheses (SURVIVED / REJECTED)
6. Three evidence-backed problems with the dataset
7. Conclusion & limitations


> **Standalone note.** This notebook runs in Jupyter *and* Google Colab. The first code cell loads `train.csv` locally if present, otherwise it downloads the public Kaggle-training mirror automatically — no setup needed.

## 1. Load and profile the dataset

We load the raw `train.csv` without modifying it, then inspect its shape, columns, types, identifiers,
and missingness. We do **not** yet compute any survival rates — profiling only.

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Works both inside the repo and standalone (e.g. Google Colab):
# try local copies first, else download the public Kaggle-train mirror.
CANDIDATES = ["../data/train.csv", "data/train.csv", "train.csv"]
MIRROR = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:
    urllib.request.urlretrieve(MIRROR, "train.csv")   # 891-row Titanic training file
    path = "train.csv"
print("Loaded Titanic data from:", path)

RAW = pd.read_csv(path)   # untouched original
df = RAW.copy()           # we work on a copy
print("shape:", df.shape)
df.head()

In [ ]:
# Schema, types, identifiers
print("Columns & dtypes:")
print(df.dtypes)
print("\nUnique PassengerId:", df["PassengerId"].nunique(), "of", len(df),
      "-> identifiers are unique" if df["PassengerId"].nunique() == len(df) else "-> DUPLICATES")
print("Survived labels:", sorted(df["Survived"].unique()))
print("Sex:", df["Sex"].unique().tolist())
print("Pclass:", sorted(df["Pclass"].unique()))
print("Embarked:", df["Embarked"].dropna().unique().tolist())

In [ ]:
# Missingness profile (counts and share)
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "share": (df.isna().mean() * 100).round(1)
})
miss[miss["missing"] > 0].sort_values("missing", ascending=False)

**Profiling observations (no outcomes used yet).**
- 891 rows, 12 columns, one row per passenger, `PassengerId` is unique.
- `Survived` is a clean 0/1 label; `Sex`, `Pclass`, `Embarked` have valid categories.
- Three columns have missing values: **Age (177, ~20%)**, **Cabin (687, ~77%)**, **Embarked (2)**.
- Cabin is missing for most passengers — we already suspect it will be unusable for many comparisons.

These facts shape the hypotheses and the cleaning that follow.

## 2. Three hypotheses — written *before* exploration

These are committed **before** we compute any group survival rates or draw any outcome-based chart.
They are falsifiable and stated with operational definitions, expected direction, and the metric used.
We do **not** edit their substance after seeing the data; rejected hypotheses stay rejected.

### Defining "Jack" scientifically
Jack Dawson is fictional and has no record in the data. We study **comparable real passengers** instead,
with a transparent, pre-chosen profile:

> **Jack-like primary subgroup:** male · third class · aged **18–25** · survival observed from the data.

We chose 18–25 *before* looking at outcomes because the character is a young adult man; a tight band keeps
the comparison honest. A wider 18–30 band is used later only as a clearly labelled sensitivity check, not to
chase a more dramatic number. We do **not** assume Jack travelled alone just because a passenger has no
recorded relatives — the family columns do not capture every travelling companion.

### H1 — Sex within class
Among **adult** passengers (age ≥ 18), **women had higher survival rates than men within the same passenger
class.** Metric: survival rate (survivors / passengers), compared female vs male inside each of the three classes.
Expected direction: female > male in every class.

### H2 — Class, for Jack-like men
Among **men aged 18–25**, **first-class passengers had a higher observed survival rate than third-class
passengers.** Metric: survival rate, first vs third class. Expected direction: first > third.

### H3 — Recorded family
Among **third-class men aged 18–25**, those with **at least one recorded family member** (`SibSp + Parch > 0`)
had a higher survival rate than those with **none recorded**. Metric: survival rate, family vs no-recorded-family.
Expected direction: family > none. We treat "no recorded family" as a data label, **not** proof of travelling alone.

> *These three statements are the pre-registered hypotheses. They are not edited after exploration.*

In [ ]:
# Lock the hypotheses as data so later cells restate them verbatim.
HYPOTHESES = {
    "H1": "Among adults (>=18), women survived more than men within the same class.",
    "H2": "Among men aged 18-25, first class survived more than third class.",
    "H3": "Among third-class men 18-25, those with recorded family survived more than those with none.",
}
# Jack-like definition, fixed here and reused everywhere.
def jack_mask(d, age_lo=18, age_hi=25):
    return (d["Sex"] == "male") & (d["Pclass"] == 3) & (d["Age"].between(age_lo, age_hi))

for k, v in HYPOTHESES.items():
    print(k, "-", v)

## 3. Cleaning — with a reason for every decision

We do not blindly drop missing rows or impute ages. For each decision we record:
**Problem → Action → Reason → Consequence.** We keep an untouched `RAW` copy throughout.

In [ ]:
clean = df.copy()
log = []   # cleaning decision log

# --- Embarked: 2 missing ---
n_emb = clean["Embarked"].isna().sum()
# Action: leave as-is (we never group by Embarked in the core analysis).
log.append(("Embarked missing (2)", "Left unchanged",
            "Not used in any Jack-related comparison; imputing would invent information.",
            "No effect on core results."))

# --- Age: 177 missing (~20%) ---
# Action: do NOT impute. Use complete-case analysis wherever age is needed.
age_missing = clean["Age"].isna().sum()
log.append((f"Age missing ({age_missing}, {age_missing/len(clean):.0%})",
            "No imputation; complete-case for age-specific analysis",
            "Filling with mean/median would fabricate ages for exactly the group we study (young men); "
            "complete-case is more honest for age bands.",
            "Age-based subgroups are smaller; we always report their n."))

# --- Cabin: 687 missing (~77%) ---
log.append((f"Cabin missing ({clean['Cabin'].isna().sum()}, "
            f"{clean['Cabin'].isna().mean():.0%})",
            "Excluded from analysis (kept in the frame, not used)",
            "Too incomplete to compare fairly; missingness is tied to class (see Section 6).",
            "No cabin-based claims are made."))

# --- Duplicates / anomalies ---
dup_ids = clean["PassengerId"].duplicated().sum()
dup_names = clean["Name"].duplicated().sum()
neg_fare = (clean["Fare"] < 0).sum()
zero_fare = (clean["Fare"] == 0).sum()
log.append(("Duplicate IDs / names",
            f"Checked: {dup_ids} dup IDs, {dup_names} dup names",
            "Confirm each row is a distinct passenger.",
            "None found; no rows removed."))
log.append((f"Fare anomalies (0 fare: {zero_fare}, negative: {neg_fare})",
            "Flagged, not deleted",
            "Zero fares are real records (staff / special tickets); deleting would bias the sample.",
            "Fare interpreted cautiously; not central to the Jack question."))

pd.DataFrame(log, columns=["Problem", "Action", "Reason", "Consequence"])

In [ ]:
# A small, reusable helper: an engineered family-size feature (transparent, no imputation).
clean["FamilySize"] = clean["SibSp"] + clean["Parch"]        # recorded companions only
clean["HasRecordedFamily"] = clean["FamilySize"] > 0
# Age-known flag for honest denominators later.
clean["AgeKnown"] = clean["Age"].notna()
print("Rows kept:", len(clean), "(no rows deleted during cleaning)")
print("Age known:", clean['AgeKnown'].sum(), "| Age missing:", (~clean['AgeKnown']).sum())

**Cleaning summary.** No rows were deleted. Age is never imputed — for age-specific questions we use
**complete-case** analysis and always print the denominator. Cabin is set aside because it is 77% missing
and that missingness is not random (shown in Section 6). `FamilySize` and `HasRecordedFamily` are transparent
features built only from recorded relatives, with the explicit caveat that they do not prove companionship.

## 4. Exploration — distributions first, then relationships

We begin with the shape of the population (no outcomes), then move to survival associations and interactions.
Every chart answers a question; discarded ideas are noted rather than hidden.

### 4A. Understand the population (distributions)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
clean["Pclass"].value_counts().sort_index().plot(
    kind="bar", ax=ax[0], color="#8A8A8A")
ax[0].set_title("Passengers per class"); ax[0].set_xlabel("Pclass"); ax[0].set_ylabel("count")
clean["Sex"].value_counts().plot(kind="bar", ax=ax[1], color="#8A8A8A")
ax[1].set_title("Sex representation"); ax[1].set_xlabel("")
clean["Age"].dropna().plot(kind="hist", bins=20, ax=ax[2], color="#8A8A8A", edgecolor="white")
ax[2].set_title("Age distribution (known ages)"); ax[2].set_xlabel("Age")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
clean["Fare"].plot(kind="hist", bins=40, ax=ax[0], color="#8A8A8A", edgecolor="white")
ax[0].set_title("Fare distribution (right-skewed)"); ax[0].set_xlabel("Fare")
clean["FamilySize"].value_counts().sort_index().plot(kind="bar", ax=ax[1], color="#8A8A8A")
ax[1].set_title("Recorded family size"); ax[1].set_xlabel("SibSp + Parch")
plt.tight_layout(); plt.show()

**Population notes.** Third class is the largest group; men outnumber women; ages cluster in the 20s–30s;
fares are heavily right-skewed (a few very expensive tickets); most passengers have **no** recorded family members.
None of this uses survival yet.

### 4B. Survival associations

In [ ]:
overall = clean["Survived"].mean()
print(f"Overall survival: {overall:.1%}  ({clean['Survived'].sum()}/{len(clean)})\n")

def rate_table(group):
    g = clean.groupby(group)["Survived"].agg(n="count", survived="sum", rate="mean")
    g["rate"] = (g["rate"] * 100).round(1)
    return g

print("By sex:"); print(rate_table("Sex"), "\n")
print("By class:"); print(rate_table("Pclass"))

In [ ]:
# Sex x class, both rates and counts
rate_sc = clean.pivot_table("Survived", "Sex", "Pclass", aggfunc="mean").round(3)
cnt_sc = clean.pivot_table("Survived", "Sex", "Pclass", aggfunc="count")
print("Survival RATE by sex x class:"); print(rate_sc)
print("\nGROUP SIZE by sex x class:"); print(cnt_sc)

In [ ]:
# Survival by age band, and age band within sex
bands = pd.cut(clean["Age"], [0, 12, 18, 25, 40, 60, 120],
               labels=["0-12", "13-18", "18-25", "25-40", "40-60", "60+"])
tmp = clean.assign(AgeBand=bands)
print("Survival rate by age band (known ages):")
print((tmp.groupby("AgeBand", observed=True)["Survived"].agg(n="count", rate="mean")
       .assign(rate=lambda d: (d["rate"]*100).round(1))))

### 4C. Interactions — does sex or class dominate, and do the aggregates mislead?

In [ ]:
# Grouped bar: survival rate by sex within each class (this becomes poster chart 1).
classes = [1, 2, 3]
female = [rate_sc.loc["female", c] for c in classes]
male = [rate_sc.loc["male", c] for c in classes]
x = np.arange(3); w = 0.38
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(x - w/2, female, w, label="Women", color="#8A8A8A")
ax.bar(x + w/2, male, w, label="Men", color="#C4303B")
ax.set_xticks(x); ax.set_xticklabels(["1st", "2nd", "3rd"])
ax.set_ylabel("Survival rate"); ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Survival by sex within each class")
ax.legend(frameon=False)
plt.tight_layout()
try:
    import os
    os.makedirs("figures/exploration", exist_ok=True)
    plt.savefig("figures/exploration/sex_within_class.png", dpi=120)
except Exception:
    pass   # saving is optional when running standalone (e.g. Colab)
plt.show()

**Interaction finding (the heart of the investigation).** The large *overall* class gradient
(1st 63% → 3rd 24%) is heavily driven by **who is in each class**. Women dominate survival inside every class,
and the sex gap is enormous (e.g. 2nd class: women 92% vs men 7%). This is a Simpson-like aggregation effect:
the headline "class" story partly reflects differing **sex composition**, not class alone. We test this carefully
rather than declaring a paradox.

In [ ]:
# Men aged 18-25 by class (this becomes poster chart 2). Small n is the point.
m1825 = clean[(clean["Sex"] == "male") & (clean["Age"].between(18, 25))]
by_c = m1825.groupby("Pclass")["Survived"].agg(n="count", survived="sum", rate="mean")
by_c["rate%"] = (by_c["rate"]*100).round(1)
print("Men 18-25, survival by class:"); print(by_c[["n", "survived", "rate%"]])

Among **young men specifically**, the class gradient nearly disappears and is *non-monotonic*:
1st class 30% but on only **10 passengers**, 2nd class **4%** (the worst), 3rd class 12% on 86. The tidy
"first class saves you" story collapses to a tiny, unstable sample once we hold sex and age fixed.

## 5. Testing the three hypotheses

For each: restate verbatim, define the subset, compute rates with numerators/denominators, quantify the
difference in percentage points, consider sample size and a Wilson 95% interval, then mark
**SURVIVED** or **REJECTED / NOT SUPPORTED**. Wilson intervals are descriptive statistics, not models.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson score 95% interval for a proportion."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0, c-h), min(1, c+h))

def summarize(sub, label):
    k, n = int(sub["Survived"].sum()), len(sub)
    r = k/n if n else float("nan")
    lo, hi = wilson(k, n)
    print(f"  {label}: {k}/{n} = {r:.1%}  (95% CI {lo:.1%}-{hi:.1%})")
    return r, k, n, lo, hi

### H1 — *Among adults (≥18), women survived more than men within the same class.*

In [ ]:
ad = clean[clean["Age"] >= 18]
h1_ok = True
for c in [1, 2, 3]:
    print(f"Class {c}:")
    rf, *_ = summarize(ad[(ad.Pclass==c) & (ad.Sex=="female")], "women")
    rm, *_ = summarize(ad[(ad.Pclass==c) & (ad.Sex=="male")], "men")
    print(f"   difference: {100*(rf-rm):+.1f} pp\n")
    h1_ok = h1_ok and (rf > rm)
print("H1 verdict:", "SURVIVED" if h1_ok else "REJECTED")

**H1 → SURVIVED.** Women out-survived men in **all three** classes, by +60, +84, and +28 percentage points.
Large, consistent, and based on substantial samples. This is the most robust finding in the data.

### H2 — *Among men aged 18–25, first class survived more than third class.*

In [ ]:
m = clean[(clean.Sex=="male") & (clean.Age.between(18, 25))]
r1, k1, n1, lo1, hi1 = summarize(m[m.Pclass==1], "1st class")
r3, k3, n3, lo3, hi3 = summarize(m[m.Pclass==3], "3rd class")
print(f"\n   difference: {100*(r1-r3):+.1f} pp")
print(f"   1st-class CI {lo1:.1%}-{hi1:.1%}  vs  3rd-class CI {lo3:.1%}-{hi3:.1%}")
print("   CIs overlap heavily, and 1st class is only", n1, "passengers.")

**H2 → NOT SUPPORTED (direction consistent, evidence too weak).**
The point estimate is in the hypothesised direction (30% vs 12%, +18 pp), **but** the first-class group is just
**10 passengers** (3 survivors). Its Wilson interval (11%–60%) overlaps the third-class interval (6%–20%), and
2nd-class young men actually did *worse* (4%). We cannot conclude a real first-vs-third advantage for young men
from this sample. This is a *not-supported* result, not a conclusively disproven one.

### H3 — *Among third-class men 18–25, recorded family beat none recorded.*

In [ ]:
j = clean[jack_mask(clean)]
fam = j[j["HasRecordedFamily"]]
nofam = j[~j["HasRecordedFamily"]]
rf, kf, nf, lof, hif = summarize(fam, "recorded family")
rn, kn, nn, lon, hin = summarize(nofam, "no recorded family")
print(f"\n   difference: {100*(rf-rn):+.1f} pp")
print(f"   but recorded-family group is only n={nf} ({kf} survivors).")

**H3 → NOT SUPPORTED (sample far too small).**
Direction is as predicted (27% vs 9%, +18 pp), but the recorded-family group is **11 passengers with 3
survivors**. With a group this small, the difference is well within noise, and "no recorded family" does not
equal "travelled alone." Not enough evidence to support the claim.

### 5B. Jack-like passengers in depth, and a responsible counterfactual

We now look directly at the Jack-like group and compare it to *responsible* alternatives — groups that differ
in exactly one characteristic — always with sizes and uncertainty, never with causal language.

In [ ]:
j = clean[jack_mask(clean)]
rk, kk, nk, lok, hik = summarize(j, "Jack-like (male, 3rd, 18-25)")

print("\nResponsible comparisons (one thing changed at a time):")
print("1) Change CLASS: young men 18-25")
for c in [1, 2, 3]:
    summarize(clean[(clean.Sex=="male") & (clean.Pclass==c) & clean.Age.between(18,25)], f"   class {c}")
print("2) Change SEX: 3rd class, 18-25")
summarize(clean[(clean.Sex=="male") & (clean.Pclass==3) & clean.Age.between(18,25)], "   men")
summarize(clean[(clean.Sex=="female") & (clean.Pclass==3) & clean.Age.between(18,25)], "   women")

In [ ]:
# Sensitivity: widen age band to 18-30 (clearly labelled; does NOT replace the definition).
print("Sensitivity check, Jack-like with age 18-30 instead of 18-25:")
summarize(clean[jack_mask(clean, 18, 30)], "   male, 3rd, 18-30")
print("Conclusion stays the same: survival stays very low (~15%), so the result is robust to the age boundary.")

**Counterfactual reading (carefully).** Holding age and sex fixed, moving Jack to first class raises the
*observed* rate from 12% to 30% — but on 10 people. Holding age and class fixed, being a **woman** instead of a
man raises it from 12% to **47%** — a larger, better-supported gap. So the data point to **sex** as the clearest
disadvantage, and sex is a demographic fact, not a ticket he could have bought. Class is associated with survival
but, *for his age-sex group*, the evidence is thin. None of this proves any single change would have saved him.

### 5C. The hidden trap

We examined several candidate traps (confounding, misleading denominators, informative missingness, small-sample
illusion, misread variables, sampling coverage, correlation≠causation). The **most consequential** for *this*
question is a combination:

> **The confident but wrong conclusion "Jack should have bought a first-class ticket."**
> It is produced by (a) reading the *overall* class gradient, which is confounded by sex composition, and
> (b) ignoring that the first-class *young-male* cell is only 10 passengers. Both the confounding and the
> small-sample illusion push toward a dramatic causal claim the data cannot support.

In [ ]:
# Demonstrate the trap numerically: overall class gap vs the young-male class gap.
overall_gap = clean[clean.Pclass==1]["Survived"].mean() - clean[clean.Pclass==3]["Survived"].mean()
ym = clean[(clean.Sex=="male") & clean.Age.between(18,25)]
ym_gap = ym[ym.Pclass==1]["Survived"].mean() - ym[ym.Pclass==3]["Survived"].mean()
print(f"Overall 1st-vs-3rd survival gap:            {overall_gap:+.0%}  (large, headline-grabbing)")
print(f"Same gap among MEN aged 18-25:              {ym_gap:+.0%}  on just "
      f"{(ym.Pclass==1).sum()} vs {(ym.Pclass==3).sum()} passengers (fragile)")
print("The headline gap shrinks and rests on a tiny sample once sex and age are held fixed.")

## 6. Three evidence-backed problems with the dataset

Not "some values are missing" — we prove *what* is missing, *how much*, and *why it matters for Jack*.

### Problem 1 — Age is missing **non-randomly**, concentrated in third-class men
If age were missing at random it would not bias age-based subgroups. It is not.

In [ ]:
age_by_class = clean.groupby("Pclass")["Age"].apply(lambda s: s.isna().mean())
print("Share of AGE missing by class:")
print((age_by_class*100).round(1).astype(str) + "%")
m3 = clean[(clean.Sex=="male") & (clean.Pclass==3)]
print(f"\n3rd-class men: {len(m3)} total, age missing for {m3['Age'].isna().sum()} "
      f"({m3['Age'].isna().mean():.0%}).")
print(f"Survival of age-MISSING 3rd-class men: {m3[m3.Age.isna()]['Survived'].mean():.1%}")
print(f"Survival of age-KNOWN 3rd-class men:   {m3[m3.Age.notna()]['Survived'].mean():.1%}")

**Why it matters:** 27% of third-class men have no recorded age and they survived *less* (10%) than the
age-known ones (15%). Our Jack-like group uses only age-known passengers, so complete-case analysis may make the
Jack-like survival rate look slightly **better** than reality. We flag this rather than impute over it.

### Problem 2 — Cabin is 77% missing, and the missingness tracks class

In [ ]:
cab = clean.groupby("Pclass")["Cabin"].apply(lambda s: s.isna().mean())
print("Share of CABIN missing by class:")
print((cab*100).round(1).astype(str) + "%")

**Why it matters:** cabin (a proxy for location on the ship, potentially relevant to lifeboat access) is
97.6% missing in third class vs 18.5% in first. Any "where on the ship was Jack" analysis is impossible for his
group — the record simply isn't there, and the gap is itself a class artefact.

### Problem 3 — The Jack-like subgroup is small, so percentages are unstable

In [ ]:
for label, sub in [
    ("Jack-like (male,3rd,18-25)", clean[jack_mask(clean)]),
    ("...with recorded family", clean[jack_mask(clean) & clean.HasRecordedFamily]),
    ("1st-class men 18-25", clean[(clean.Sex=="male") & (clean.Pclass==1) & clean.Age.between(18,25)]),
]:
    k, n = int(sub["Survived"].sum()), len(sub)
    lo, hi = wilson(k, n)
    print(f"{label:32s} n={n:3d}  rate={k/n:5.1%}  95% CI [{lo:.0%}, {hi:.0%}]")

**Why it matters:** key comparison cells have 10–11 passengers. A single different outcome swings the rate
by ~10 points, and the Wilson intervals are very wide. Any confident headline built on these cells is a
small-sample illusion — the single most dangerous misreading of this dataset for our question.

## 7. Conclusion and limitations

**Could Jack have been saved, according to the available data?**

Among the 86 real passengers who most resemble Jack — men in third class aged 18–25 — only **10 survived
(12%)**. Being **male** was the clearest and most robust disadvantage: within every class women survived far more
(by 28–84 percentage points), and holding class and age fixed, comparable women survived at **47%** versus his
group's 12%. Passenger class is associated with survival overall, but *for his age-and-sex group* that advantage
rests on just **10 first-class young men** and is statistically fragile — so the tempting conclusion that "a
first-class ticket would have saved him" is exactly the trap this dataset sets.

**Verdicts:** H1 **SURVIVED**; H2 **NOT SUPPORTED** (right direction, sample too small); H3 **NOT SUPPORTED**
(sample too small). The strongest, most defensible insight is therefore about **sex**, not class.

**What the data cannot establish:**
- Any survival probability for a *fictional* individual.
- Whether changing one characteristic (ticket class) would *cause* a different outcome — these are observational
  group differences, not interventions.
- Anything about lifeboat access, evacuation timing, or location aboard the ship (not in the data; cabin is
  mostly missing for third class).
- Conclusions about *everyone* aboard — `train.csv` is a subset of the passengers, not the full manifest.

**Honest bottom line:** the data describe *who* was disadvantaged with confidence, and refuse to tell us whether
Jack, specifically, could have been saved. That refusal is the finding.